# LeetCode 287 — Find the Duplicate Number bằng Python

Link đề: https://leetcode.com/problems/find-the-duplicate-number/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers (Floyd), Binary Search, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def findDuplicate(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (đếm tần suất bằng `Counter`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `nums` có `n + 1` số nguyên, mỗi số thuộc `[1, n]`; **đúng một** giá trị lặp lại (có thể lặp **hai lần trở lên**); tìm giá trị đó **mà không sửa mảng** và chỉ dùng **bộ nhớ phụ hằng số**.
2. Chứng minh được bằng **nguyên lý chuồng bồ câu** rằng luôn tồn tại số lặp (câu hỏi follow-up).
3. Đi từ **vét cạn** `O(n²)` → **sắp xếp** → **bảng băm (set)** → **chặt nhị phân theo giá trị** `O(n log n)`, `O(1)` bộ nhớ → **đếm bit** `O(n log n)` → **thuật toán Floyd (rùa và thỏ) trên "đồ thị hàm"** `O(n)`, `O(1)`.
4. Hiểu cách **biến mảng thành danh sách liên kết có chu trình**: `i → nums[i]`; vị trí **bắt đầu chu trình** chính là số lặp.
5. Tránh các lỗi kinh điển: sắp xếp/sửa mảng (vi phạm đề), dùng set (vi phạm bộ nhớ), dùng công thức tổng/XOR (chỉ đúng khi số lặp xuất hiện đúng hai lần và không thiếu giá trị), chặt nhị phân trên **chỉ số** thay vì **giá trị**, và cài sai Floyd (do-while, quên đặt lại con trỏ).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` gồm `n + 1` phần tử, mỗi phần tử trong khoảng `[1, n]` (đóng).

Chỉ có **một số lặp** duy nhất trong `nums`; hãy trả về số đó. Bạn phải giải **mà không sửa** mảng `nums` và chỉ dùng **bộ nhớ phụ hằng số**.

**Ví dụ 1**

- Input: `nums = [1,3,4,2,2]`
- Output: `2`

**Ví dụ 2**

- Input: `nums = [3,1,3,4,2]`
- Output: `3`

**Ví dụ 3**

- Input: `nums = [3,3,3,3,3]`
- Output: `3`

**Ràng buộc:** `1 <= n <= 10^5`; `nums.length == n + 1`; `1 <= nums[i] <= n`; mọi số xuất hiện **một lần**, trừ **đúng một** số xuất hiện **từ hai lần trở lên**.

**Follow up:** (a) chứng minh luôn có ít nhất một số lặp; (b) giải trong thời gian **tuyến tính**.

In [1]:
from typing import List
from collections import Counter
import random
import time

examples = [
    ([1, 3, 4, 2, 2], 2),
    ([3, 1, 3, 4, 2], 3),
    ([3, 3, 3, 3, 3], 3),
]
for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[1, 3, 4, 2, 2] -> kỳ vọng 2
nums=[3, 1, 3, 4, 2] -> kỳ vọng 3
nums=[3, 3, 3, 3, 3] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Nguyên lý chuồng bồ câu

Có `n + 1` phần tử nhưng chỉ có `n` giá trị khả dĩ `1..n` ("chuồng"). Nếu mọi giá trị đều khác nhau, ta chỉ dùng tối đa `n` giá trị ⇒ chỉ có `n` phần tử, mâu thuẫn. Vậy **ít nhất một giá trị xuất hiện từ hai lần trở lên** — đó là lời giải cho câu hỏi chứng minh (a).
#### Code demo

In [2]:
n = 4
nums = [1, 2, 3, 4, 4]                       # n + 1 = 5 phần tử, giá trị trong [1, 4]
print(len(nums), "phần tử vs", len(set(nums)), "giá trị khác nhau <= n =", n)
assert len(set(nums)) <= n

5 phần tử vs 4 giá trị khác nhau <= n = 4


### 3.2. Số lặp có thể xuất hiện nhiều hơn hai lần

Đề nói "**một** số lặp" nhưng số đó có thể xuất hiện **2, 3, ..., n + 1 lần** (ví dụ 3: `[3,3,3,3,3]`). Khi đó có **giá trị thiếu**. Vì thế các mẹo như `sum(nums) - n(n+1)/2` hay XOR **chỉ đúng khi số lặp xuất hiện đúng hai lần**.
#### Code demo

In [3]:
nums = [3, 1, 3, 4, 2]            # lặp đúng hai lần, không thiếu giá trị nào
print(sum(nums) - 4 * 5 // 2)     # 3: đúng (hiệu tổng = số lặp)
nums = [3, 3, 3, 3, 3]            # lặp 5 lần, thiếu 1, 2, 4
print(sum(nums) - 4 * 5 // 2)     # 5: SAI (đáp án là 3)

3
5


### 3.3. Chặt nhị phân trên **giá trị** (không phải trên chỉ số)

Mảng **không sắp xếp** nên không chặt nhị phân trên chỉ số được. Nhưng **không gian giá trị `[1, n]`** có thể bị chặt đôi: với `mid`, đếm `count(x <= mid)`. Nếu mọi giá trị trong `[1, mid]` chỉ xuất hiện tối đa một lần thì `count <= mid`. Nếu `count > mid` thì (chuồng bồ câu) trong `[1, mid]` có số lặp; ngược lại số lặp nằm trong `[mid + 1, n]`.
#### Code demo

In [4]:
nums = [1, 3, 4, 2, 2]
for mid in range(1, 5):
    count = sum(1 for x in nums if x <= mid)
    print(f"mid={mid}: có {count} phần tử <= {mid} -> {'> mid: số lặp trong [1, ' + str(mid) + ']' if count > mid else '<= mid: số lặp trong [' + str(mid + 1) + ', 4]'}")

mid=1: có 1 phần tử <= 1 -> <= mid: số lặp trong [2, 4]
mid=2: có 3 phần tử <= 2 -> > mid: số lặp trong [1, 2]
mid=3: có 4 phần tử <= 3 -> > mid: số lặp trong [1, 3]
mid=4: có 5 phần tử <= 4 -> > mid: số lặp trong [1, 4]


### 3.4. Dãy `i → nums[i]` như một danh sách liên kết

Xem mỗi chỉ số `i ∈ [0, n]` là một **nút** và `nums[i]` là **nút kế tiếp** (con trỏ `next`). Vì `nums[i] ∈ [1, n]`, **không nút nào trỏ tới nút `0`**, nên bắt đầu từ `0` ta không thể quay về `0`; mà chỉ có `n + 1` nút nên đi mãi sẽ phải **lặp lại một nút** ⇒ có **chu trình**. Hai chỉ số khác nhau cùng có `nums[i] = v` ⇒ nút `v` có **hai nút cùng trỏ vào** — chỗ **chu trình bắt đầu**. Vậy **số lặp = nút đầu vào của chu trình**.
#### Code demo

In [5]:
nums = [1, 3, 4, 2, 2]
path, i = [], 0
for _ in range(8):
    path.append(i)
    i = nums[i]
print(path)       # [0, 1, 3, 2, 4, 2, 4, 2]: đi vào chu trình (2 -> 4 -> 2 ...), đầu vào chu trình là 2 = số lặp

[0, 1, 3, 2, 4, 2, 4, 2]


### 3.5. Thuật toán Floyd (rùa và thỏ)

Trên danh sách có chu trình: **rùa** đi 1 bước, **thỏ** đi 2 bước mỗi lượt; chắc chắn gặp nhau trong chu trình. Sau đó đặt một con trỏ về đầu, cả hai cùng đi **1 bước mỗi lượt**; chúng **gặp nhau đúng tại nút đầu vào chu trình**. Chứng minh ở mục 10.
#### Code demo

In [6]:
def meet_in_cycle(nums):
    slow = fast = 0
    while True:                                    # do-while: phải đi ít nhất một lần vì ban đầu hai con trỏ bằng nhau
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            return slow


print("điểm gặp trong chu trình:", meet_in_cycle([1, 3, 4, 2, 2]))

điểm gặp trong chu trình: 4


### 3.6. Đếm bit

Số lặp `d` có thể tìm **từng bit**: với mỗi vị trí bit `b`, so sánh `c1` = số phần tử của `nums` có bit `b` bằng `1` với `c2` = số giá trị trong `1..n` có bit `b` bằng `1`. Nếu bit `b` của `d` là `1` thì `c1 > c2` (có dư các bản sao của `d`), ngược lại `c1 <= c2`. Chứng minh ở mục 10.
#### Code demo

In [7]:
nums, n = [1, 3, 4, 2, 2], 4
for b in range(n.bit_length()):
    c1 = sum((x >> b) & 1 for x in nums)
    c2 = sum((x >> b) & 1 for x in range(1, n + 1))
    print(f"bit {b}: c1={c1}, c2={c2} -> bit của số lặp = {int(c1 > c2)}")

bit 0: c1=2, c2=2 -> bit của số lặp = 0
bit 1: c1=3, c2=2 -> bit của số lặp = 1
bit 2: c1=1, c2=1 -> bit của số lặp = 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`nums = [1, 3, 4, 2, 2]` (`n = 4`). Đi theo `i → nums[i]` từ `0`:

`0 → 1 → 3 → 2 → 4 → 2 → 4 → ...`

Từ nút `2` ta vào chu trình `2 → 4 → 2`. Nút `2` được trỏ tới **hai lần** (từ `3` và từ `4`) ⇒ `2` là số lặp. Cách nghĩ: mảng không sửa được và không thêm bộ nhớ được, nhưng ta **có thể đi dọc** `i → nums[i]` bằng chỉ vài biến; "số lặp" biến thành bài **tìm đầu chu trình** (bài *Linked List Cycle II*).

## 5. Cách 1 — Brute Force / baseline: so từng cặp

**Ý tưởng.** Hai vòng lặp: nếu `nums[i] == nums[j]` với `i < j` thì trả `nums[i]`. Không sửa mảng, không dùng bộ nhớ phụ nhưng chậm.
#### Code demo

In [8]:
def duplicate_bruteforce(nums):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] == nums[j]:
                return nums[i]
    return -1


for nums, expected in examples:
    print(nums, "->", duplicate_bruteforce(nums), "(kỳ vọng", expected, ")")

[1, 3, 4, 2, 2] -> 2 (kỳ vọng 2 )
[3, 1, 3, 4, 2] -> 3 (kỳ vọng 3 )
[3, 3, 3, 3, 3] -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Gọi `N = n + 1`: **Time `O(N²)`** (với `n = 10^5` là `~5·10^9` phép — quá chậm), **Space `O(1)`**.

**Nút thắt.** So sánh từng cặp. Nhiều cách nhanh hơn nhưng phải **vi phạm** một trong hai ràng buộc (sắp xếp/sửa mảng, hoặc dùng bộ nhớ phụ `O(n)`).

## 6. Ý tưởng thuật toán / bước đột phá

Ta cần **thời gian tốt** mà **không sửa mảng** và **`O(1)` bộ nhớ**. Có hai hướng độc đáo:

1. **Tìm kiếm theo giá trị:** chặt nhị phân trên đoạn giá trị `[1, n]` bằng phép **đếm** (`count(x <= mid)`), `O(n log n)`, `O(1)`.
2. **Đồ thị hàm + Floyd:** coi `i → nums[i]` là con trỏ ⇒ có chu trình, đầu vào chu trình là số lặp ⇒ **`O(n)` thời gian, `O(1)` bộ nhớ**.

Cũng có thể xác định số lặp **từng bit** bằng đếm (`O(n log n)`, `O(1)`).

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp rồi so hai phần tử kề

**Giải thích.** `sorted(nums)` (bản sao) rồi tìm hai phần tử liền kề bằng nhau. **Vi phạm** ràng buộc bộ nhớ nếu dùng bản sao (và vi phạm "không sửa" nếu sắp tại chỗ), nhưng thường dùng để so sánh.
#### Code demo

In [9]:
def duplicate_sort(nums):
    a = sorted(nums)
    for i in range(1, len(a)):
        if a[i] == a[i - 1]:
            return a[i]
    return -1


for nums, expected in examples:
    print(nums, "->", duplicate_sort(nums), "(kỳ vọng", expected, ")")

[1, 3, 4, 2, 2] -> 2 (kỳ vọng 2 )
[3, 1, 3, 4, 2] -> 3 (kỳ vọng 3 )
[3, 3, 3, 3, 3] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 2:** **Time `O(n log n)`**, **Space `O(n)`** (bản sao).

### Cách 3 — Tập hợp (set)

**Giải thích.** Duyệt từng phần tử; nếu đã có trong tập thì trả; không thì thêm. Đơn giản nhất nhưng tốn bộ nhớ.
#### Code demo

In [10]:
def duplicate_set(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return x
        seen.add(x)
    return -1


for nums, expected in examples:
    print(nums, "->", duplicate_set(nums), "(kỳ vọng", expected, ")")

[1, 3, 4, 2, 2] -> 2 (kỳ vọng 2 )
[3, 1, 3, 4, 2] -> 3 (kỳ vọng 3 )
[3, 3, 3, 3, 3] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 4 — Chặt nhị phân trên giá trị (đếm)

**Giải thích.** `lo, hi = 1, n`. Lặp `lo < hi`: `mid = (lo + hi) // 2`, `count = số phần tử <= mid`. Nếu `count > mid` thì số lặp trong `[lo, mid]`, đặt `hi = mid`; ngược lại trong `[mid + 1, hi]`, đặt `lo = mid + 1`. Khi `lo == hi` đó là đáp án.
#### Code demo

In [11]:
def duplicate_binary_search(nums):
    lo, hi = 1, len(nums) - 1                        # n = len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        count = sum(1 for x in nums if x <= mid)     # đếm số phần tử có giá trị <= mid
        if count > mid:
            hi = mid                                 # chuồng bồ câu: có số lặp trong [lo, mid]
        else:
            lo = mid + 1                             # ngược lại số lặp trong [mid + 1, hi]
    return lo


for nums, expected in examples:
    print(nums, "->", duplicate_binary_search(nums), "(kỳ vọng", expected, ")")

[1, 3, 4, 2, 2] -> 2 (kỳ vọng 2 )
[3, 1, 3, 4, 2] -> 3 (kỳ vọng 3 )
[3, 3, 3, 3, 3] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (`log n` vòng chặt, mỗi vòng đếm `O(n)`), **Space `O(1)`**, **không sửa mảng**.

### Cách 5 — Đếm bit

**Giải thích.** Với mỗi bit `b` từ `0` đến `n.bit_length() - 1`: `c1` = số phần tử `nums` có bit `b`; `c2` = số giá trị trong `1..n` có bit `b`; nếu `c1 > c2` thì bật bit `b` của đáp án.
#### Code demo

In [12]:
def duplicate_bits(nums):
    n = len(nums) - 1
    answer = 0
    for b in range(n.bit_length()):
        c1 = sum((x >> b) & 1 for x in nums)
        c2 = sum((x >> b) & 1 for x in range(1, n + 1))
        if c1 > c2:
            answer |= 1 << b
    return answer


for nums, expected in examples:
    print(nums, "->", duplicate_bits(nums), "(kỳ vọng", expected, ")")

[1, 3, 4, 2, 2] -> 2 (kỳ vọng 2 )
[3, 1, 3, 4, 2] -> 3 (kỳ vọng 3 )
[3, 3, 3, 3, 3] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 5:** **Time `O(n log n)`** (`~17` bit × `O(n)`), **Space `O(1)`**, không sửa mảng.

### Cách 6 — Floyd (rùa và thỏ), cách nộp chuẩn

**Giải thích.** Pha 1: `slow = nums[slow]`, `fast = nums[nums[fast]]` (bắt đầu từ `0`) cho tới khi `slow == fast` (gặp trong chu trình). Pha 2: đặt `slow2 = 0`; cả `slow` và `slow2` đi **một bước** mỗi lượt cho tới khi bằng nhau; giá trị đó là đáp án.
#### Code demo

In [13]:
def duplicate_floyd(nums):
    slow = fast = 0
    while True:                                      # pha 1: gặp nhau trong chu trình
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    slow2 = 0                                        # pha 2: tìm đầu vào chu trình
    while slow != slow2:
        slow = nums[slow]
        slow2 = nums[slow2]
    return slow


for nums, expected in examples:
    print(nums, "->", duplicate_floyd(nums), "(kỳ vọng", expected, ")")

[1, 3, 4, 2, 2] -> 2 (kỳ vọng 2 )
[3, 1, 3, 4, 2] -> 3 (kỳ vọng 3 )
[3, 3, 3, 3, 3] -> 3 (kỳ vọng 3 )


In [14]:
# Code nộp LeetCode cho Cách 6
class Solution:
    def findDuplicate(self, nums: List[int]) -> int:
        slow = fast = 0
        while True:
            slow = nums[slow]
            fast = nums[nums[fast]]
            if slow == fast:
                break
        slow = 0
        while slow != fast:
            slow = nums[slow]
            fast = nums[fast]
        return slow


print(Solution().findDuplicate([1, 3, 4, 2, 2]))   # 2

2


**Độ phức tạp Cách 6:** **Time `O(n)`** (pha 1 và pha 2 đều tối đa `O(n)` bước), **Space `O(1)`**, không sửa mảng — đáp ứng mọi ràng buộc kể cả follow-up (b).

> Cách 6 là bản nộp tối ưu. Cách 4 là lựa chọn dễ nghĩ ra nhất khi chưa biết Floyd. Cách 5 minh họa tư duy "bit".

## 8. TRACE từng bước

Ta trace Floyd trên `[1, 3, 4, 2, 2]` (cả hai pha) và chặt nhị phân trên `[3, 1, 3, 4, 2]`.
#### Code demo

In [15]:
def trace_floyd(nums):
    print("nums =", nums)
    print("đồ thị hàm i -> nums[i]:", {i: v for i, v in enumerate(nums)})
    slow = fast = 0
    step = 0
    print("pha 1:")
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        step += 1
        print(f"  lượt {step}: slow={slow}, fast={fast}")
        if slow == fast:
            print("  gặp nhau tại", slow)
            break
    print("pha 2: đặt p = 0, cả hai đi 1 bước/lượt")
    p = 0
    step = 0
    while slow != p:
        print(f"  lượt {step}: slow={slow}, p={p}")
        slow = nums[slow]
        p = nums[p]
        step += 1
    print(f"  gặp nhau tại {slow} -> số lặp = {slow}")
    return slow


trace_floyd([1, 3, 4, 2, 2])

nums = [1, 3, 4, 2, 2]
đồ thị hàm i -> nums[i]: {0: 1, 1: 3, 2: 4, 3: 2, 4: 2}
pha 1:
  lượt 1: slow=1, fast=3
  lượt 2: slow=3, fast=4
  lượt 3: slow=2, fast=4
  lượt 4: slow=4, fast=4
  gặp nhau tại 4
pha 2: đặt p = 0, cả hai đi 1 bước/lượt
  lượt 0: slow=4, p=0
  lượt 1: slow=2, p=1
  lượt 2: slow=4, p=3
  gặp nhau tại 2 -> số lặp = 2


2

In [16]:
def trace_binary_search(nums):
    print("nums =", nums)
    lo, hi = 1, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        count = sum(1 for x in nums if x <= mid)
        verdict = f"count={count} > mid={mid} -> hi=mid" if count > mid else f"count={count} <= mid={mid} -> lo=mid+1"
        print(f"  [lo={lo}, hi={hi}] mid={mid}: {verdict}")
        if count > mid:
            hi = mid
        else:
            lo = mid + 1
    print("  số lặp =", lo)
    return lo


trace_binary_search([3, 1, 3, 4, 2])

nums = [3, 1, 3, 4, 2]
  [lo=1, hi=4] mid=2: count=2 <= mid=2 -> lo=mid+1
  [lo=3, hi=4] mid=3: count=4 > mid=3 -> hi=mid
  số lặp = 3


3

Bảng tóm tắt Floyd (`[1,3,4,2,2]`):

| bước | `slow` | `fast` | ghi chú |
|---|---|---|---|
| bắt đầu | 0 | 0 | |
| 1 | 1 | 3 | `slow = nums[0]`, `fast = nums[nums[0]]` |
| 2 | 3 | 4 | |
| 3 | 2 | 4 | `fast` đi `4 → 2 → 4` |
| 4 | 4 | 4 | **gặp nhau tại 4** (trong chu trình `2 ↔ 4`) |
| pha 2 | `slow = 4`, `p = 0` | | cả hai đi 1 bước: `(4, 0) → (2, 1) → (4, 3) → (2, 2)`: gặp nhau tại `2` |

Pha 2 kết thúc tại nút `2` — chính là số lặp.

## 9. Code nộp LeetCode

Bản nộp dùng Floyd (Cách 6). Đúng `class Solution`, đúng `findDuplicate(self, nums: List[int]) -> int`, không sửa `nums`, `O(1)` bộ nhớ.

In [17]:
class Solution:
    def findDuplicate(self, nums: List[int]) -> int:
        slow = fast = 0                          # pha 1: rùa 1 bước, thỏ 2 bước, gặp nhau trong chu trình
        while True:
            slow = nums[slow]
            fast = nums[nums[fast]]
            if slow == fast:
                break
        slow = 0                                 # pha 2: từ đầu và từ điểm gặp cùng đi 1 bước
        while slow != fast:
            slow = nums[slow]
            fast = nums[fast]
        return slow                              # nút đầu vào chu trình = số lặp


sol = Solution()
for nums, expected in examples:
    assert sol.findDuplicate(nums) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Có chu trình và đầu vào chu trình là số lặp:** đồ thị `i → nums[i]` trên `n + 1` nút, mọi cạnh trỏ tới `[1, n]` nên nút `0` không có cạnh vào ⇒ đi từ `0` mãi sẽ lặp một nút ⇒ chu trình. Nút đầu vào chu trình `c` có **hai** cạnh vào (một từ đường dẫn vào, một từ chu trình) ⇒ có hai chỉ số `i ≠ j` với `nums[i] = nums[j] = c` ⇒ `c` là số lặp. Vì chỉ có **một** giá trị lặp nên `c` là nó.
- **Floyd pha 1:** gọi `a` là độ dài đoạn từ `0` đến `c` (số bước), `L` là độ dài chu trình, và rùa/thỏ gặp nhau tại điểm cách `c` đúng `k` bước trong chu trình. Khi rùa đi `a + k` bước, thỏ đi `2(a + k)` bước và hai bước đi này khác nhau một số nguyên lần `L`: `2(a + k) − (a + k) = a + k = m·L` với `m ≥ 1` nguyên.
- **Floyd pha 2:** từ `a + k = mL` suy ra `a = mL − k`: đi từ điểm gặp thêm `a` bước (trong chu trình) tới `c` (vì đi `mL − k` bước từ điểm cách `c` là `k` bước dọc chu trình — tức lùi về `c`). Con trỏ từ `0` đi `a` bước cũng tới `c`. Hai con trỏ cùng đi 1 bước mỗi lượt **gặp nhau đúng tại `c`**.
- **Chặt nhị phân:** khẳng định `count(x <= mid) > mid ⟺ d <= mid` (với `d` là số lặp xuất hiện `k ≥ 2` lần; tổng cộng có `k − 2` giá trị thiếu). Nếu `d <= mid`: số phần tử `<= mid` bằng (số giá trị phân biệt `<= mid` có mặt) `+ (k − 1)` bản sao thừa của `d`; số giá trị phân biệt có mặt `>= mid − (k − 2)`, nên `count >= mid − (k − 2) + (k − 1) = mid + 1 > mid`. Nếu `d > mid`: mọi phần tử `<= mid` là các giá trị **phân biệt** trong `[1, mid]` nên `count <= mid`. Vậy `count > mid` thì giữ nửa `[lo, mid]`, ngược lại giữ `[mid + 1, hi]`.
- **Đếm bit:** `c1 − c2 = (k − 1)·bit_b(d) − Σ_{giá trị thiếu} bit_b(m)`, với `k ≥ 2` là số lần xuất hiện của `d` và có đúng `k − 2` giá trị thiếu. Nếu `bit_b(d) = 1` thì `c1 − c2 ≥ (k − 1) − (k − 2) = 1 > 0`; nếu `bit_b(d) = 0` thì `c1 − c2 = −Σ ≤ 0`. Vậy `c1 > c2 ⟺ bit_b(d) = 1`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là cận giá trị, `N = n + 1` số phần tử.

| Cách | Time | Space | Sửa mảng? |
|---|---|---|---|
| 1. Vét cạn | `O(N²)` | `O(1)` | không |
| 2. Sắp xếp | `O(N log N)` | `O(N)` (bản sao) | không (nếu dùng `sorted`) |
| 3. Set | `O(N)` | `O(N)` | không |
| 4. Chặt nhị phân giá trị | `O(N log n)` | `O(1)` | không |
| 5. Đếm bit | `O(N log n)` | `O(1)` | không |
| 6. Floyd | `O(N)` | `O(1)` | không |

Với `n = 10^5`: Floyd cỡ `3·10^5` bước, chặt nhị phân cỡ `1.7·10^6`, đếm bit cỡ `3.4·10^6`, vét cạn `~5·10^9`. Chỉ Cách 4–6 thỏa **cả** hai ràng buộc (không sửa, `O(1)` bộ nhớ); chỉ Cách 6 thỏa follow-up tuyến tính.

## 12. Bộ test

`oracle` là brute force độc lập: dùng `Counter` (không dùng bất kỳ cách nào ở trên) để lấy giá trị xuất hiện nhiều hơn một lần, và kiểm tra tính hợp lệ của đầu vào (chỉ một giá trị lặp, mọi giá trị trong `[1, n]`). Sinh đầu vào ngẫu nhiên với số lặp xuất hiện `k` lần (`k ≥ 2`) và các giá trị còn lại phân biệt.
#### Code demo

In [18]:
def oracle(nums):
    counts = Counter(nums)
    repeated = [v for v, c in counts.items() if c > 1]
    n = len(nums) - 1
    assert len(repeated) == 1 and all(1 <= v <= n for v in nums)
    return repeated[0]


def make_input(n, d, k, rng):
    """n + 1 phần tử trong [1, n]; giá trị d xuất hiện đúng k lần (2 <= k <= n + 1), các giá trị khác phân biệt."""
    others = [v for v in range(1, n + 1) if v != d]
    chosen = rng.sample(others, n + 1 - k)
    nums = [d] * k + chosen
    rng.shuffle(nums)
    return nums


impls = {
    "bruteforce": duplicate_bruteforce,
    "sort": duplicate_sort,
    "set": duplicate_set,
    "binary_search": duplicate_binary_search,
    "bits": duplicate_bits,
    "floyd": duplicate_floyd,
    "submit": sol.findDuplicate,
}

for nums, expected in examples:
    assert oracle(nums) == expected
    for key, fn in impls.items():
        before = list(nums)
        assert fn(nums) == expected, (key, nums, fn(nums), expected)
        assert nums == before, (key, "đã sửa mảng!")
print("PASS ví dụ của đề (mọi cách và oracle, không cách nào sửa mảng)")

PASS ví dụ của đề (mọi cách và oracle, không cách nào sửa mảng)


### 12.1. Các trường hợp biên
#### Code demo

In [19]:
edge_cases = [
    ([1, 1], 1, "n = 1: mảng nhỏ nhất"),
    ([1, 1, 2], 1, "n = 2, lặp ở đầu"),
    ([2, 1, 2], 2, "n = 2, lặp ở hai đầu"),
    ([2, 2, 2], 2, "n = 2, lặp 3 lần"),
    ([1, 2, 3, 4, 4], 4, "lặp ở cuối, giá trị lớn nhất"),
    ([4, 4, 1, 2, 3], 4, "lặp ở đầu"),
    ([1, 4, 4, 2, 3], 4, "lặp liên tiếp"),
    ([3, 3, 3, 3, 3], 3, "ví dụ lặp n + 1 lần"),
    ([2, 5, 9, 6, 9, 3, 8, 9, 7, 1], 9, "lặp ba lần"),
    ([1, 3, 4, 2, 2], 2, "ví dụ của đề"),
    ([2, 2, 2, 2, 2], 2, "toàn bằng nhau, số khác 1"),
    ([1, 1, 1, 1, 1], 1, "toàn 1 (nút 1 trỏ về chính nó: Floyd vẫn đúng)"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == expected, (label, oracle(nums), expected)
    for key, fn in impls.items():
        before = list(nums)
        assert fn(nums) == expected, (key, label, nums, fn(nums), expected)
        assert nums == before
    print("PASS", label)

PASS n = 1: mảng nhỏ nhất
PASS n = 2, lặp ở đầu
PASS n = 2, lặp ở hai đầu
PASS n = 2, lặp 3 lần
PASS lặp ở cuối, giá trị lớn nhất
PASS lặp ở đầu
PASS lặp liên tiếp
PASS ví dụ lặp n + 1 lần
PASS lặp ba lần
PASS ví dụ của đề
PASS toàn bằng nhau, số khác 1
PASS toàn 1 (nút 1 trỏ về chính nó: Floyd vẫn đúng)


### 12.2. Tính chất và phản ví dụ

- Kết quả luôn thuộc `[1, n]` và xuất hiện `≥ 2` lần trong `nums`.
- **Bất biến thứ tự:** xáo trộn `nums` không đổi kết quả của Cách 4, 5, 3, 2, 1 (riêng Floyd phụ thuộc cấu trúc nhưng vẫn cho đúng đáp án).
- **Phản ví dụ** cho công thức tổng / XOR: `[3,3,3,3,3]` cho `5` (tổng) thay vì `3`.
#### Code demo

In [20]:
rng = random.Random(287)
for _ in range(500):
    n = rng.randint(1, 30)
    d = rng.randint(1, n)
    k = rng.randint(2, n + 1)
    nums = make_input(n, d, k, rng)
    ans = sol.findDuplicate(nums)
    assert ans == d and nums.count(ans) == k and 1 <= ans <= n
    shuffled = nums[:]
    rng.shuffle(shuffled)
    assert sol.findDuplicate(shuffled) == d
print("PASS tính chất (kết quả thuộc [1, n], xuất hiện k lần, bất biến thứ tự) trên 500 mảng ngẫu nhiên")


def sai_cong_thuc_tong(nums):
    n = len(nums) - 1
    return sum(nums) - n * (n + 1) // 2


print("phản ví dụ [3,3,3,3,3]: đúng", sol.findDuplicate([3, 3, 3, 3, 3]), "| công thức tổng:", sai_cong_thuc_tong([3, 3, 3, 3, 3]))
assert sai_cong_thuc_tong([3, 3, 3, 3, 3]) != 3

PASS tính chất (kết quả thuộc [1, n], xuất hiện k lần, bất biến thứ tự) trên 500 mảng ngẫu nhiên
phản ví dụ [3,3,3,3,3]: đúng 3 | công thức tổng: 5


### 12.3. Đối chiếu ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [21]:
rng = random.Random(4287)
for trial in range(3000):
    n = rng.randint(1, 40)
    d = rng.randint(1, n)
    k = rng.choice([2, 2, 2, 3, rng.randint(2, n + 1)])
    k = min(k, n + 1)
    nums = make_input(n, d, k, rng)
    truth = oracle(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == truth, (key, nums, fn(nums), truth)
print("PASS 3000 mảng ngẫu nhiên (số lặp xuất hiện 2, 3 hoặc nhiều lần): mọi cách khớp oracle")

# n = 10^5: bỏ vét cạn O(N^2); so các cách còn lại
n = 100_000
for d, k in [(1, 2), (n, 2), (50_000, 2), (777, 3), (12_345, 1000), (n // 2, n + 1)]:
    nums = make_input(n, d, k, rng)
    row = []
    for key in ("sort", "set", "binary_search", "bits", "floyd", "submit"):
        t0 = time.perf_counter()
        value = impls[key](nums)
        dt = (time.perf_counter() - t0) * 1000
        assert value == d, (key, d, k, value)
        row.append(f"{key}={dt:7.1f}ms")
    print(f"d={d:>6}, k={k:>6}: " + " ".join(row))
print("PASS n = 10^5")

PASS 3000 mảng ngẫu nhiên (số lặp xuất hiện 2, 3 hoặc nhiều lần): mọi cách khớp oracle


d=     1, k=     2: sort=   18.4ms set=    8.7ms binary_search=   47.2ms bits=  152.6ms floyd=    8.0ms submit=    8.1ms


d=100000, k=     2: sort=   20.7ms set=    7.7ms binary_search=   55.1ms bits=  171.9ms floyd=   12.2ms submit=   10.5ms


d= 50000, k=     2: sort=   20.9ms set=    8.7ms binary_search=   67.4ms bits=  170.4ms floyd=   11.3ms submit=   11.7ms


d=   777, k=     3: sort=   16.9ms set=    3.2ms binary_search=   46.9ms bits=  152.5ms floyd=   12.9ms submit=   12.6ms


d= 12345, k=  1000: sort=   17.7ms set=    0.0ms binary_search=   54.9ms bits=  153.7ms floyd=    0.0ms submit=    0.0ms
d= 50000, k=100001: sort=    0.6ms set=    0.0ms binary_search=   27.9ms bits=  142.9ms floyd=    0.0ms submit=    0.0ms
PASS n = 10^5


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp xếp tại chỗ (sửa mảng đầu vào)
#### Code sai

In [22]:
def sai_sap_xep_tai_cho(nums):
    nums.sort()                                     # SAI: đề cấm sửa nums
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1]:
            return nums[i]


arr = [3, 1, 3, 4, 2]
print("kết quả:", sai_sap_xep_tai_cho(arr), "| mảng sau khi gọi:", arr, "(đã bị đổi)")

kết quả: 3 | mảng sau khi gọi: [1, 2, 3, 3, 4] (đã bị đổi)


**Vì sao sai:** đề yêu cầu **không sửa** `nums`. Sắp xếp tại chỗ vi phạm; dùng bản sao thì vi phạm bộ nhớ `O(1)`.

### Lỗi 2: Dùng công thức tổng hoặc XOR
Xem `sai_cong_thuc_tong` ở phần test: chỉ đúng khi số lặp xuất hiện **đúng hai lần** và các giá trị còn lại đủ; sai với `[3,3,3,3,3]`. **Code đúng:** Floyd / chặt nhị phân / đếm bit không giả định số lần lặp.

### Lỗi 3: Floyd — dùng `while slow != fast` ngay từ đầu (không chạy lần nào)
#### Code sai

In [23]:
def sai_floyd_khong_do_while(nums):
    slow = fast = 0
    while slow != fast:                             # SAI: ban đầu slow == fast nên vòng lặp bị bỏ qua
        slow = nums[slow]
        fast = nums[nums[fast]]
    slow2 = 0
    while slow != slow2:
        slow = nums[slow]
        slow2 = nums[slow2]
    return slow


print("[1,3,4,2,2] đúng:", sol.findDuplicate([1, 3, 4, 2, 2]), "| sai:", sai_floyd_khong_do_while([1, 3, 4, 2, 2]))

[1,3,4,2,2] đúng: 2 | sai: 0


**Vì sao sai:** hai con trỏ cùng bắt đầu từ `0` nên điều kiện `slow != fast` sai ngay từ đầu. Phải dùng **do-while**: đi một bước trước rồi mới so sánh (`while True: ...; if slow == fast: break`).

### Lỗi 4: Floyd — quên đặt lại một con trỏ về `0` ở pha 2
#### Code sai

In [24]:
def sai_floyd_khong_dat_lai(nums):
    slow = fast = 0
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    return slow                                     # SAI: đây là điểm gặp trong chu trình, chưa chắc là đầu vào chu trình


for nums in ([1, 3, 4, 2, 2], [3, 1, 3, 4, 2]):
    print(nums, "đúng:", sol.findDuplicate(nums), "| sai (điểm gặp):", sai_floyd_khong_dat_lai(nums))

[1, 3, 4, 2, 2] đúng: 2 | sai (điểm gặp): 4
[3, 1, 3, 4, 2] đúng: 3 | sai (điểm gặp): 2


**Vì sao sai:** điểm gặp của rùa và thỏ nằm **ở đâu đó trong chu trình**, không nhất thiết ở đầu vào. Pha 2 (một con trỏ từ `0`, một từ điểm gặp, cùng đi 1 bước) mới đưa về đầu vào chu trình.

### Lỗi 5: Chặt nhị phân trên chỉ số thay vì trên giá trị
#### Code sai

In [25]:
def sai_chat_nhi_phan_chi_so(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] == nums[mid + 1]:               # SAI: mảng chưa sắp xếp nên so kề không có ý nghĩa
            return nums[mid]
        lo = mid + 1
    return nums[lo]


for nums in ([1, 3, 4, 2, 2], [3, 1, 3, 4, 2]):
    print(nums, "đúng:", sol.findDuplicate(nums), "| sai:", sai_chat_nhi_phan_chi_so(nums))

[1, 3, 4, 2, 2] đúng: 2 | sai: 2
[3, 1, 3, 4, 2] đúng: 3 | sai: 2


**Vì sao sai:** mảng **không sắp xếp** nên không có tính đơn điệu theo chỉ số. Cái đơn điệu là **số phần tử `<= mid` theo giá trị `mid`**: chặt nhị phân trên đoạn giá trị `[1, n]`.

### Lỗi 6: Dùng `count >= mid` thay vì `count > mid`
#### Code sai

In [26]:
def sai_dem_lon_hon_hoac_bang(nums):
    lo, hi = 1, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        count = sum(1 for x in nums if x <= mid)
        if count >= mid:                             # SAI: count == mid là bình thường (mọi giá trị 1..mid có mặt đúng một lần)
            hi = mid
        else:
            lo = mid + 1
    return lo


for nums in ([1, 3, 4, 2, 2], [2, 5, 9, 6, 9, 3, 8, 9, 7, 1]):
    print(nums, "đúng:", sol.findDuplicate(nums), "| sai:", sai_dem_lon_hon_hoac_bang(nums))

[1, 3, 4, 2, 2] đúng: 2 | sai: 1
[2, 5, 9, 6, 9, 3, 8, 9, 7, 1] đúng: 9 | sai: 9


**Vì sao sai:** `count == mid` xảy ra khi mỗi giá trị `1..mid` có đúng một bản — **không** có số lặp trong `[1, mid]`. Chỉ `count > mid` mới chứng tỏ có.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**`n + 1` số trong `[1, n]`**, tìm số lặp, **không sửa mảng, `O(1)` bộ nhớ**" → **Floyd** (coi mảng là danh sách liên kết) hoặc **chặt nhị phân theo giá trị**.
- Dạng `a[i] ∈ [0, n]` hay `[1, n]` đóng vai **con trỏ** → nghĩ ngay **đồ thị hàm / chu trình** (*Linked List Cycle II*, *Happy Number*).
- Floyd nhớ 3 điểm: **do-while** ở pha 1, **đặt lại về đầu** ở pha 2, rồi **cả hai đi 1 bước**.
- Chặt nhị phân **không nhất thiết trên chỉ số**: có thể trên **giá trị** với hàm đếm đơn điệu.
- Các mẹo tổng/XOR chỉ dùng khi số lặp xuất hiện **đúng hai lần** và không thiếu giá trị.
- Họ hàng: **Find All Duplicates in an Array** (442), **First Missing Positive** (41), **Missing Number** (268), **Linked List Cycle II** (142), **Happy Number** (202).

## 15. Biến thể của bài

1. **Find All Duplicates in an Array** (LeetCode 442): nhiều số lặp, mỗi số tối đa hai lần — đánh dấu bằng dấu âm (cho phép sửa).
2. **Find All Numbers Disappeared in an Array** (448).
3. **Missing Number** (268): thiếu một số trong `0..n`.
4. **First Missing Positive** (41): dùng mảng làm bảng băm.
5. **Linked List Cycle II** (142): tìm nút đầu vào chu trình — chính là pha 2 của Floyd.
#### Code demo

In [27]:
def find_all_duplicates(nums):
    """LeetCode 442 (cho phép sửa mảng): mỗi giá trị trong [1, n] xuất hiện tối đa hai lần."""
    nums = nums[:]
    result = []
    for x in nums:
        idx = abs(x) - 1
        if nums[idx] < 0:
            result.append(abs(x))
        else:
            nums[idx] = -nums[idx]
    return result


def missing_number(nums):
    n = len(nums)
    x = n
    for i, v in enumerate(nums):
        x ^= i ^ v
    return x


print(find_all_duplicates([4, 3, 2, 7, 8, 2, 3, 1]))     # [2, 3]
print(missing_number([3, 0, 1]))                         # 2

[2, 3]
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Sửa mảng? | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Vét cạn | so từng cặp | `O(N²)` | `O(1)` | không | oracle |
| 2. Sắp xếp | so hai phần tử kề | `O(N log N)` | `O(N)` | không (bản sao) | dễ nghĩ |
| 3. Set | thêm/kiểm tra tập | `O(N)` | `O(N)` | không | thực dụng khi không ràng buộc bộ nhớ |
| 4. Chặt nhị phân giá trị | đếm `count(x <= mid)` | `O(N log n)` | `O(1)` | không | thỏa ràng buộc, dễ chứng minh |
| 5. Đếm bit | so số bit 1 của `nums` và `1..n` | `O(N log n)` | `O(1)` | không | tư duy bit |
| 6. Floyd | rùa & thỏ trên `i → nums[i]` | `O(N)` | `O(1)` | không | **tối ưu, bản nộp chuẩn** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) chuỗi chỉ số khi đi `i → nums[i]` từ `0` trên `nums = [2, 5, 9, 6, 9, 3, 8, 9, 7, 1]` (6 bước đầu); (b) `Solution().findDuplicate([1, 1])`; (c) `Solution().findDuplicate([2, 2, 2])`.

**Bài 2 (tính bằng tay).** Chạy tay Floyd trên `[3, 1, 3, 4, 2]` và ghi hai pha; sau đó chạy tay chặt nhị phân.

**Bài 3 (điền TODO).** Hoàn thiện `dup_todo(nums)` theo Floyd.
#### Code demo

In [28]:
def dup_todo(nums):
    slow = fast = 0
    # TODO 1: pha 1: while True: slow = nums[slow]; fast = nums[nums[fast]]; nếu slow == fast thì break
    # TODO 2: pha 2: slow2 = 0; while slow != slow2: slow = nums[slow]; slow2 = nums[slow2]
    # TODO 3: trả slow
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples:
#     assert dup_todo(nums) == expected

**Bài 4 (tự viết lại).** Viết `dup_by_counting_sort(nums)` dùng mảng đếm kích thước `n + 1` (cho phép bộ nhớ phụ) và so với Floyd; sau đó viết `dup_binary_search2` **không nhìn lại** Cách 4.

**Bài 5 (biến thể / mini challenge).** Viết `find_cycle_start(next_of, start)` tìm đầu chu trình của một hàm `next_of: int -> int` bằng Floyd, rồi dùng nó để giải **Happy Number** (LeetCode 202: `n` là số vui nếu quá trình "thay bằng tổng bình phương các chữ số" kết thúc ở `1`; thử `19 → True`, `2 → False`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: đi theo `i → nums[i]`. Bài 2: pha 1 gặp nhau trong chu trình, pha 2 về `0`. Bài 5: coi `next_of(x)` là con trỏ.
- **Mức 2 (vừa):** Bài 1: `0 → 2 → 9 → 1 → 5 → 3 → 6 → ...`; `1`; `2`. Bài 2: `0→3→4→2→3...`: chu trình `3 → 4 → 2 → 3`. Bài 3: khung Floyd.
- **Mức 3 (sâu):** Bài 2: pha 1 gặp nhau (tính bằng code để kiểm tra), pha 2 về `3`. Bài 5: Happy Number: chuỗi `next_of` hoặc kết thúc ở `1` (điểm bất động) hoặc vào chu trình khác `1`; đầu chu trình `== 1` ⇒ vui.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
chain, i = [], 0
arr = [2, 5, 9, 6, 9, 3, 8, 9, 7, 1]
for _ in range(6):
    chain.append(i)
    i = arr[i]
assert chain == [0, 2, 9, 1, 5, 3]
assert Solution().findDuplicate([1, 1]) == 1 and Solution().findDuplicate([2, 2, 2]) == 2

# Bài 2
assert sol.findDuplicate([3, 1, 3, 4, 2]) == 3 == duplicate_binary_search([3, 1, 3, 4, 2])


# Bài 3
def dup_todo_ans(nums):
    slow = fast = 0
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    slow2 = 0
    while slow != slow2:
        slow = nums[slow]
        slow2 = nums[slow2]
    return slow


for nums, expected in examples:
    assert dup_todo_ans(nums) == expected


# Bài 4
def dup_by_counting(nums):
    seen = [0] * len(nums)
    for x in nums:
        seen[x] += 1
        if seen[x] == 2:
            return x


def dup_binary_search2(nums):
    lo, hi = 1, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(1 for x in nums if x <= mid) > mid:
            hi = mid
        else:
            lo = mid + 1
    return lo


rng = random.Random(18)
for _ in range(300):
    n = rng.randint(1, 25)
    nums = make_input(n, rng.randint(1, n), rng.randint(2, n + 1), rng)
    assert dup_by_counting(nums) == dup_binary_search2(nums) == dup_todo_ans(nums) == oracle(nums)


# Bài 5
def find_cycle_start(next_of, start):
    slow = fast = start
    while True:
        slow = next_of(slow)
        fast = next_of(next_of(fast))
        if slow == fast:
            break
    slow = start
    while slow != fast:
        slow = next_of(slow)
        fast = next_of(fast)
    return slow


def is_happy(n):
    def step(x):
        return sum(int(ch) ** 2 for ch in str(x))

    return find_cycle_start(step, n) == 1


assert is_happy(19) is True and is_happy(2) is False and is_happy(1) is True
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Floyd: rùa và thỏ)

```python
class Solution:
    def findDuplicate(self, nums: List[int]) -> int:
        slow = fast = 0
        while True:
            slow = nums[slow]
            fast = nums[nums[fast]]
            if slow == fast:
                break
        slow = 0
        while slow != fast:
            slow = nums[slow]
            fast = nums[fast]
        return slow
```

**Checklist**

- [ ] Hiểu `i → nums[i]` là con trỏ; nút `0` không có cạnh vào ⇒ có chu trình; **đầu vào chu trình = số lặp**?
- [ ] Pha 1 dùng **do-while** (`while True` + `break`), thỏ đi **2 bước**?
- [ ] Pha 2: đặt một con trỏ về `0`, cả hai đi **1 bước** tới khi gặp nhau?
- [ ] Không sửa `nums`, không dùng bộ nhớ phụ `O(n)`?
- [ ] Biết phương án dự phòng: chặt nhị phân theo **giá trị** (`count > mid`) và đếm bit?
- [ ] Không dùng công thức tổng/XOR (sai khi lặp ≥ 3 lần)?
- [ ] Đã thử `[1,1]`, `[3,3,3,3,3]`, lặp ở đầu/cuối, `n = 10^5`?
- [ ] Skeleton đúng `class Solution` / `def findDuplicate(self, nums: List[int]) -> int`?